# Activity 1: Cross-Tool Refresher Drills (SQL vs Pandas)

**Module:** Week 6 Day 1
**Estimated Time:** 40 to 50 minutes
**Format:** Individual; every drill runs in Snowflake first, then you match it in pandas

## Objective

Warm up for Week 6 with a two-window workflow: Snowsight on one side, this notebook on the other. Each drill gives you working SQL. You run it in Snowflake, look at the result, then make pandas produce the **exact same output**.

Drills 1 to 3 are Week 4 SQL you know cold. Drills 4 to 6 use SQL you have **not** met yet (`LAG`, moving averages, `DENSE_RANK`). That is deliberate: run them anyway, watch them work, and know that they are today's main topic. For those three, the notebook first teaches you the pandas method on a tiny example, then hands you the real problem.

Copy this notebook to `student-work/week6/day1/` before you start.

## Setup, part 1: the table in Snowflake

Copy this into a Snowsight worksheet and run it once. Twelve Hartford-style claims, small enough to check every answer by hand.

```sql
USE ROLE DE;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE TECHCATALYST;
USE SCHEMA TECHCATALYST.<YOUR_NAME>;

CREATE OR REPLACE TRANSIENT TABLE W5D1_KICKOFF_CLAIMS (
  claim_id    VARCHAR(10),
  region      VARCHAR(20),
  line        VARCHAR(10),
  claim_month VARCHAR(7),
  paid_amount NUMBER(10, 0),
  open_days   NUMBER(5, 0)
);

INSERT INTO W5D1_KICKOFF_CLAIMS VALUES
  ('C-1001', 'Northeast', 'Auto', '2026-01',  4200, 12),
  ('C-1002', 'Northeast', 'Home', '2026-01', 12500, 45),
  ('C-1003', 'Southeast', 'Auto', '2026-01',  3100,  9),
  ('C-1004', 'Southeast', 'Home', '2026-02',  9800, 60),
  ('C-1005', 'Midwest',   'Auto', '2026-02',  5200, 21),
  ('C-1006', 'Midwest',   'Home', '2026-02', 15200, 75),
  ('C-1007', 'Northeast', 'Auto', '2026-03',  4700, 15),
  ('C-1008', 'Southeast', 'Auto', '2026-03',  2900,  8),
  ('C-1009', 'Midwest',   'Auto', '2026-03',  6100, 30),
  ('C-1010', 'Northeast', 'Home', '2026-04', 11300, 52),
  ('C-1011', 'Southeast', 'Home', '2026-04',  8600, 41),
  ('C-1012', 'Midwest',   'Home', '2026-04', 13900, 66);

SELECT COUNT(*) AS num_claims FROM W5D1_KICKOFF_CLAIMS;  -- 12
```

## Setup, part 2: the same data in pandas

Run this cell. Same 12 rows, now in a DataFrame. Two copies of one dataset, two engines, and your job all morning is proving they agree.

In [ ]:
import pandas as pd
import numpy as np

claims = pd.DataFrame({
    "claim_id": ["C-1001", "C-1002", "C-1003", "C-1004", "C-1005", "C-1006",
                 "C-1007", "C-1008", "C-1009", "C-1010", "C-1011", "C-1012"],
    "region": ["Northeast", "Northeast", "Southeast", "Southeast", "Midwest", "Midwest",
               "Northeast", "Southeast", "Midwest", "Northeast", "Southeast", "Midwest"],
    "line": ["Auto", "Home", "Auto", "Home", "Auto", "Home",
             "Auto", "Auto", "Auto", "Home", "Home", "Home"],
    "claim_month": ["2026-01", "2026-01", "2026-01", "2026-02", "2026-02", "2026-02",
                    "2026-03", "2026-03", "2026-03", "2026-04", "2026-04", "2026-04"],
    "paid_amount": [4200, 12500, 3100, 9800, 5200, 15200, 4700, 2900, 6100, 11300, 8600, 13900],
    "open_days": [12, 45, 9, 60, 21, 75, 15, 8, 30, 52, 41, 66],
})

claims

### Setup, part 3: the same data in Polars

Polars is a blazingly fast, multi-threaded DataFrame library. Convert the `claims` DataFrame into a Polars DataFrame:

In [ ]:
import polars as pl

claims_pl = pl.DataFrame(claims)
claims_pl

## Drill 1: WHERE and ORDER BY

**Run in Snowflake:**

```sql
SELECT claim_id, region, paid_amount
FROM W5D1_KICKOFF_CLAIMS
WHERE line = 'Auto' AND paid_amount > 4000
ORDER BY paid_amount DESC;
```

**Now match it in pandas.** Filter, sort, keep the three columns.

**Checkpoint:** 4 rows, C-1009 (6100) first, C-1001 (4200) last, identical to your Snowsight grid.

In [ ]:
# YOUR CODE: reproduce the SQL result exactly


### Polars Challenge 1
Match Drill 1 using **Polars**!
*Hint:* Use `claims_pl.filter((pl.col("line") == "Auto") & (pl.col("paid_amount") > 4000)).sort("paid_amount", descending=True).select(["claim_id", "region", "paid_amount"])

In [ ]:
# YOUR POLARS CODE HERE


## Drill 2: GROUP BY

**Run in Snowflake:**

```sql
SELECT region,
       COUNT(*)         AS num_claims,
       SUM(paid_amount) AS total_paid,
       AVG(paid_amount) AS avg_paid
FROM W5D1_KICKOFF_CLAIMS
GROUP BY region;
```

**Match it in pandas** with `groupby` and `agg`.

**Checkpoint:** Midwest 40400 total (avg 10100), Northeast 32700 (avg 8175), Southeast 24400 (avg 6100), 4 claims each.

In [ ]:
# YOUR CODE


### Polars Challenge 2
Match Drill 2 `GROUP BY` using **Polars**!
*Hint:* Use `claims_pl.group_by("region").agg(pl.len().alias("num_claims"), pl.col("paid_amount").sum().alias("total_paid"), pl.col("paid_amount").mean().alias("avg_paid"))`

In [ ]:
# YOUR POLARS CODE HERE


## Drill 3: HAVING

**Run in Snowflake:**

```sql
SELECT region, AVG(paid_amount) AS avg_paid
FROM W5D1_KICKOFF_CLAIMS
GROUP BY region
HAVING AVG(paid_amount) > 8000;
```

**Match it in pandas.** `HAVING` has no pandas keyword; it is just a filter applied **after** the groupby, which is exactly what `HAVING` means in SQL too.

**Checkpoint:** two regions survive, Midwest (10100.0) and Northeast (8175.0).

In [ ]:
# YOUR CODE


### Polars Challenge 3
Match Drill 3 `HAVING` using **Polars**!
*Hint:* Chain `.filter(pl.col("avg_paid") > 8000)` onto your `group_by("region").agg(...)` expression.

In [ ]:
# YOUR POLARS CODE HERE


## Drill 4: LAG (new SQL: run it anyway)

**Run in Snowflake:**

```sql
WITH monthly AS (
  SELECT claim_month, SUM(paid_amount) AS total_paid
  FROM W5D1_KICKOFF_CLAIMS
  GROUP BY claim_month
)
SELECT claim_month,
       total_paid,
       LAG(total_paid) OVER (ORDER BY claim_month) AS prev_total,
       total_paid - LAG(total_paid) OVER (ORDER BY claim_month) AS change
FROM monthly
ORDER BY claim_month;
```

The CTE is Week 4. The `LAG(...) OVER (...)` is not: it reached into the **previous row**. Look at the output: each month sees the month before it, and the first month sees NULL because it has no previous. This is a window function, and it is the heart of today. For now, park the SQL and learn the pandas spelling.

### Meet `shift`

`shift(1)` slides a whole column down one row, which is exactly "the previous row's value". Run this tiny example:

In [ ]:
demo = pd.Series([10, 20, 35, 40])
pd.DataFrame({"value": demo, "previous": demo.shift(1), "change": demo - demo.shift(1)})

First row gets NaN (no previous row, same as SQL's NULL), every other row sees its predecessor.

### Your turn

Reproduce the SQL output: build the monthly grain first (`groupby` on `claim_month`, sum, like the CTE did), then add `prev_total` and `change` with `shift`.

**Checkpoint:** monthly totals 19800, 30200, 13700, 33800; change NaN, +10400, -16500, +20100, matching your Snowsight grid.

In [ ]:
# YOUR CODE: monthly grain, then shift


### Polars Challenge 4
Match Drill 4 `LAG` / shift using **Polars**!
*Hint:* Use `pl.col("total_paid").shift(1)` inside `.with_columns()` on your monthly aggregated DataFrame.

In [ ]:
# YOUR POLARS CODE HERE


## Drill 5: Moving average (new SQL: run it anyway)

**Run in Snowflake** (continues from the same `monthly` CTE):

```sql
WITH monthly AS (
  SELECT claim_month, SUM(paid_amount) AS total_paid
  FROM W5D1_KICKOFF_CLAIMS
  GROUP BY claim_month
)
SELECT claim_month,
       total_paid,
       ROUND(AVG(total_paid) OVER (
         ORDER BY claim_month
         ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ), 2) AS moving_avg_3m
FROM monthly
ORDER BY claim_month;
```

Read the frame out loud: "this row and the two before it, averaged." Notice the first month still gets a value: SQL averages whatever rows exist so far.

### Meet `rolling`

`rolling(3)` builds the same sliding 3-row window. One difference to see for yourself:

In [ ]:
demo = pd.Series([10, 20, 35, 40])
pd.DataFrame({
    "value": demo,
    "rolling_3": demo.rolling(3).mean(),
    "rolling_3_min1": demo.rolling(3, min_periods=1).mean().round(2),
})

Default `rolling(3)` waits for a full window of 3 and gives NaN before that. `min_periods=1` averages whatever exists so far, which is exactly the SQL behavior you just saw in Snowsight.

### Your turn

Add `moving_avg_3m` to your monthly DataFrame so it **matches the SQL output exactly** (so: which of the two variants do you need?). Round to 2.

**Checkpoint:** 19800.0, 25000.0, 21233.33, 25900.0, identical to Snowsight.

In [ ]:
# YOUR CODE


### Polars Challenge 5
Match Drill 5 Moving Average using **Polars**!
*Hint:* Use `pl.col("total_paid").rolling_mean(window_size=3, min_samples=1).round(2).alias("moving_avg_3m")` on your monthly Polars DataFrame.

In [ ]:
# YOUR POLARS CODE HERE


## Drill 6: DENSE_RANK (new SQL: run it anyway)

**Run in Snowflake:**

```sql
SELECT claim_id,
       paid_amount,
       DENSE_RANK() OVER (ORDER BY paid_amount DESC) AS paid_rank
FROM W5D1_KICKOFF_CLAIMS
ORDER BY paid_rank;
```

Every row got a position, biggest claim first. `DENSE_RANK` is one of a family (`RANK`, `ROW_NUMBER`); the differences matter only when values tie, and this afternoon's activities dig into exactly that.

### Meet `rank`

Run the tiny example and note what the tie does:

In [ ]:
demo = pd.Series([90, 80, 80, 50])
pd.DataFrame({
    "value": demo,
    "dense": demo.rank(method="dense", ascending=False),
    "min_rank": demo.rank(method="min", ascending=False),
})

Both 80s share rank 2. `dense` gives the next value rank 3 (no gap); `min` jumps to 4 (a gap). SQL's `DENSE_RANK` and `RANK` split exactly the same way.

### Your turn

Add `paid_rank` to `claims` matching the SQL (`DENSE_RANK`, highest paid first) and show the top 3 rows by rank.

**Checkpoint:** C-1006 (15200) rank 1, C-1012 (13900) rank 2, C-1002 (12500) rank 3.

In [ ]:
# YOUR CODE


### Polars Challenge 6
Match Drill 6 `DENSE_RANK` using **Polars**!
*Hint:* Use `pl.col("paid_amount").rank(method="dense", descending=True).alias("paid_rank")`.

In [ ]:
# YOUR POLARS CODE HERE


## Drill 7: CASE

**Run in Snowflake:**

```sql
SELECT severity, COUNT(*) AS num_claims
FROM (
  SELECT CASE WHEN paid_amount >= 10000 THEN 'high'
              WHEN paid_amount >= 5000  THEN 'medium'
              ELSE 'low' END AS severity
  FROM W5D1_KICKOFF_CLAIMS
)
GROUP BY severity;
```

**Match it in pandas:** build a `severity` column (`np.where`, nested, or `np.select`), then count per bucket.

**Checkpoint:** exactly 4 high, 4 medium, 4 low.

In [ ]:
# YOUR CODE


### Polars Challenge 7
Match Drill 7 SQL `CASE WHEN` using **Polars**!
*Hint:* Polars has native `pl.when(...).then(...).otherwise(...)` syntax:
`claims_pl.with_columns(pl.when(pl.col("paid_amount") >= 10000).then(pl.lit("high")).when(pl.col("paid_amount") >= 5000).then(pl.lit("medium")).otherwise(pl.lit("low")).alias("severity")).group_by("severity").len()`

In [ ]:
# YOUR POLARS CODE HERE


## Wrap up

Say each of these out loud to a partner, one sentence each:

1. Drills 1 to 3: same answers from two engines. What guaranteed that?
2. Drill 4: what does `LAG ... OVER (ORDER BY ...)` do to a row, and what is its pandas spelling?
3. Drill 5: when SQL's moving average and `rolling(3)` disagreed, which rows differed and why?
4. `LAG`, the moving-average frame, and `DENSE_RANK` all had `OVER (...)` in them. That clause is the day's topic; what do you think it declares?

If your grids match your DataFrames, you are warmed up, and you have already run every window function the afternoon will formalize.